In [8]:
import pandas as pd
import numpy as np
import re
import os
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

from nltk.util import ngrams
from konlpy.tag import Okt
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

# Windows Jupyter 기준 한글 폰트
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("환경 설정 완료")

환경 설정 완료


In [9]:
def make_bi_token(tokens):
    bi_tokens = list(ngrams(tokens, 2))
    return tokens + [' '.join(bi) for bi in bi_tokens]

In [10]:
df = pd.read_csv('./data/naver_blog_contents_육아제품구독.txt',sep=r'\|\|')# || 를 정규식 escape

print(f"로딩 완료: {df.shape}")
df.head()

로딩 완료: (943, 3)


,크롤링워드,url,content
0,육아제품구독,https://m.blog.naver.com/hivahome/224201214420,​ 6개월 남자아이 키우고 있는 현실 육아대디에요. 이번에 KB국민은행 우리아이 첫...
1,육아제품구독,https://m.blog.naver.com/tjdls1817/224153011397,"​ 육아 인플루언서, Class 101 후기 도움이 되었을까? ​ ​ ​ 안녕하세요..."
2,육아제품구독,https://m.blog.naver.com/chaing88/224213127676,안녕하세요. 김부장입니다. ​ 오늘은 제가 구매한 육아용품 중에 가장 비싼 육아용품...
3,육아제품구독,https://m.blog.naver.com/alslrjf369/224211432228,오늘은 지난 11월 받았던 1차 육아 용품 패키지에 이어 2차 육아 용품 패키지가 ...
4,육아제품구독,https://m.blog.naver.com/soravan/223763203013,웅진씽크빅에서 제품을 지원받아 솔직하게 작성한 후기입니다. 책육아 웅진씽크빅 책다른...


In [11]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    
    # 한글, 공백만 남기기
    text = re.sub(r'[^가-힣\s]', '', str(text))
    
    # 여러 공백 -> 단일 공백
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content_clean'] = df['content'].apply(clean_text)

# 짧은 문장 제거
df = df[df['content_clean'].str.len() >= 10].reset_index(drop=True)

print(f"전처리 완료: {len(df)}개 데이터 남음")

전처리 완료: 943개 데이터 남음


In [12]:
okt = Okt()

stop_words_df = pd.read_csv('./data/ko-stopwords_블로그합.csv', encoding='utf-8')
stop_words = list(stop_words_df['stopwords'])

def pos_tagging(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    
    pos_words = okt.pos(text, stem=True, norm=True)
    
    tagged = [
        token for token, tag in pos_words
        if tag in ['Noun', 'Adjective', 'Verb']
        and token not in stop_words
        and len(token) > 1
    ]
    return tagged

df['tagged_review'] = df['content_clean'].apply(pos_tagging)

print("형태소 분석 완료")

JVMNotFoundException: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.

In [2]:
import pandas as pd

stopwords = pd.read_csv('./data/ko-stopwords_모든걸합.csv', encoding='utf-8')
stopwords

,stopwords
0,가
1,가까스로
2,가령
3,각
4,각각
...,...
623,가능
624,장난감
625,생각
626,필요
